In [25]:
import json
import requests

def get_weather(city):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": city,
        "count": 1
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if data['results']:
            lat = data['results'][0]['latitude']
            lon = data['results'][0]['longitude']

    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        temperature = data['current_weather']['temperature']
        return f"The weather in {city} is {temperature}°C."

In [26]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

In [27]:
from dotenv import load_dotenv
import os
import requests

load_dotenv(dotenv_path="../.env")
api_key = os.getenv("GROQ_API_KEY")

url = "https://api.groq.com/openai/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

messages = [
    {"role": "user", "content": "What is the weather in Lahore?"}
]

response = requests.post(url, headers=headers, json={
    "model": "openai/gpt-oss-20b",
    "messages": messages,
    "tools": tools,
    "temperature": 0.7
})

print(response.json())

{'id': 'chatcmpl-d2aa207f-a1d0-4b81-8940-34eb9f046d74', 'object': 'chat.completion', 'created': 1782628170, 'model': 'openai/gpt-oss-20b', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'reasoning': 'We need to call get_weather function with city "Lahore".', 'tool_calls': [{'id': 'fc_f50b9142-ba61-4d8c-94db-aff9b31157c2', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"city":"Lahore"}'}}]}, 'logprobs': None, 'finish_reason': 'tool_calls'}], 'usage': {'queue_time': 0.003147265, 'prompt_tokens': 134, 'prompt_time': 0.006442989, 'completion_tokens': 40, 'completion_time': 0.040483235, 'total_tokens': 174, 'total_time': 0.046926224, 'completion_tokens_details': {'reasoning_tokens': 15}}, 'usage_breakdown': None, 'system_fingerprint': 'fp_c5a89987dc', 'x_groq': {'id': 'req_01kw6epeqze4dsvrhyb5e4d6da', 'seed': 746676684}, 'service_tier': 'on_demand'}


In [28]:
data = response.json()
message = data["choices"][0]["message"]

if message.get("tool_calls"):
    tool_call = message["tool_calls"][0]
    function_name = tool_call["function"]["name"]
    arguments = json.loads(tool_call["function"]["arguments"])
    
    print(f"LLM wants to call: {function_name}")
    print(f"With arguments: {arguments}")

LLM wants to call: get_weather
With arguments: {'city': 'Lahore'}


In [29]:
import json

# call the actual function
result = get_weather(arguments["city"])
print(f"Function returned: {result}")

Function returned: The weather in Lahore is 36.5°C.


In [30]:
messages.append(message)  # add the LLM's tool call request to history

messages.append({
    "role": "tool",
    "tool_call_id": tool_call["id"],
    "content": result
})

final_response = requests.post(url, headers=headers, json={
    "model": "openai/gpt-oss-20b",
    "messages": messages,
    "tools": tools,
    "temperature": 0.7
})

final_answer = final_response.json()["choices"][0]["message"]["content"]
print(f"Final answer: {final_answer}")

Final answer: The current weather in Lahore is **36.5 °C**.


In [31]:
import json
print(json.dumps(messages, indent=4))

[
    {
        "role": "user",
        "content": "What is the weather in Lahore?"
    },
    {
        "role": "assistant",
        "reasoning": "We need to call get_weather function with city \"Lahore\".",
        "tool_calls": [
            {
                "id": "fc_f50b9142-ba61-4d8c-94db-aff9b31157c2",
                "type": "function",
                "function": {
                    "name": "get_weather",
                    "arguments": "{\"city\":\"Lahore\"}"
                }
            }
        ]
    },
    {
        "role": "tool",
        "tool_call_id": "fc_f50b9142-ba61-4d8c-94db-aff9b31157c2",
        "content": "The weather in Lahore is 36.5\u00b0C."
    }
]


In [32]:
print(get_weather("Lahore"))

The weather in Lahore is 36.5°C.


In [ ]:
import json
import requests
from dotenv import load_dotenv
import os


load_dotenv(dotenv_path="../.env")
api_key = os.getenv("GROQ_API_KEY")

url = "https://api.groq.com/openai/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

tools = [
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city"
                }
            },
            "required": ["city"]
        }
    }
}
]

def agent(user_input):

    messages = [{"role": "user", "content": user_input}]
    response = requests.post(url, headers=headers, json={
        "model": "openai/gpt-oss-20b",
        "messages": messages,
        "tools": tools,
        "temperature": 0.7
    })


    data = response.json()
    message = data["choices"][0]["message"]

    if message.get("tool_calls"):
        tool_call = message["tool_calls"][0]
        function_name = tool_call["function"]["name"]
        arguments = json.loads(tool_call["function"]["arguments"])

        # call the actual function
        result = get_weather(arguments["city"])

        messages.append(message)  # add the LLM's tool call request to history

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call["id"],
            "content": result
        })

        final_response = requests.post(url, headers=headers, json={
            "model": "openai/gpt-oss-20b",
            "messages": messages,
            "tools": tools,
            "temperature": 0.7
        })

        final_answer = final_response.json()["choices"][0]["message"]["content"]

        return final_response.json()["choices"][0]["message"]["content"]

    else:
        return message["content"]

In [36]:
print(agent("What is the weather in Karachi?"))
print(agent("What is the capital of Pakistan?"))

Function returned: The weather in Karachi is 31.8°C.
The current weather in Karachi is **31.8 °C**.
The capital of Pakistan is **Islamabad**.
